# 1D Triple-Well Potential: Asymmetric Configurations

Implements Section 3.3.3 of the thesis. Covers exact stationary PDFs for the asymmetric potential ($d=0.3$), direct training across 8 seeds, and transfer learning from symmetric and asymmetric source models. Reproduces Figures 3.11–3.15.

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

sys.path.append(os.path.abspath('..'))

from src.problems.triple_well import TripleWellProblem
from src.models import FPNet
from src.training import train_model
from src.utils import acc_L2
from src.losses import fp_loss

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Exact Stationary PDFs: Effect of Asymmetry Parameter $d$

Surveys the exact stationary PDF for $\sigma = 1.0$ across a range of asymmetry values $d$ and for fixed $d \in \{0.3, 0.7\}$ across $\sigma \in \{1.0, 1.5, 2.0\}$.

In [ ]:
x_eval = np.linspace(-4.0, 4.0, 1000)

D_VALUES = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5]
SIGMAS   = [1.0, 1.5, 2.0]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

print("σ=1.0 | Peak analysis:")
print(f"{'d':>5} | {'h_left':>8} | {'h_center':>9} | {'h_right':>9} | {'visible peaks':>14} | note")
print("-" * 75)

for ax, d in zip(axes, D_VALUES):
    try:
        prob = TripleWellProblem(a=0.2, b=1.5, c=3.0, d=d, sigma=1.0)
        p    = prob.exact_solution(x_eval)

        # Peak heights in the three zones
        n = len(x_eval)
        t = n // 3
        h_left   = p[:t].max()
        h_center = p[t:2*t].max()
        h_right  = p[2*t:].max()
        h_max    = max(h_left, h_center, h_right)

        visible = sum([
            h_left   > 0.05 * h_max,
            h_center > 0.05 * h_max,
            h_right  > 0.05 * h_max,
        ])

        note = ''
        if visible < 3:
            note = '<-- fewer than 3 visible peaks'

        print(f"{d:>5.1f} | {h_left:>8.4f} | {h_center:>9.4f} | "
              f"{h_right:>9.4f} | {visible:>14} | {note}")

        ax.plot(x_eval, p, 'b-', lw=2)
        ax.set_title(f'd = {d} | {visible} visible peaks', fontsize=11)
        ax.set_xlabel('x', fontsize=10)
        ax.set_ylabel('p(x)', fontsize=10)
        ax.grid(True, alpha=0.3)

    except Exception as e:
        ax.set_title(f'd = {d} | ERROR', fontsize=11)
        print(f"d={d}: ERROR — {e}")

plt.suptitle('Triple-Well Exact PDF | σ=1.0 | Varying asymmetry parameter d', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('asym_step1_sigma10.png', dpi=100, bbox_inches='tight')
plt.show()

# ─── Figure 2: Fix d=0.3 and d=0.7, vary σ ───────────────────────────────────
# (Useful for selecting the source task for Transfer Learning in Step 4)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

print("\nFixed d, varying σ:")
print(f"{'d':>5} | {'σ':>5} | {'h_left':>8} | {'h_center':>9} | {'h_right':>9}")
print("-" * 55)

for row, d in enumerate([0.3, 0.7]):
    for col, sigma in enumerate(SIGMAS):
        ax = axes[row, col]
        try:
            prob = TripleWellProblem(a=0.2, b=1.5, c=3.0, d=d, sigma=sigma)
            p    = prob.exact_solution(x_eval)

            n = len(x_eval)
            t = n // 3
            h_left   = p[:t].max()
            h_center = p[t:2*t].max()
            h_right  = p[2*t:].max()

            print(f"{d:>5.1f} | {sigma:>5.1f} | {h_left:>8.4f} | "
                  f"{h_center:>9.4f} | {h_right:>9.4f}")

            ax.plot(x_eval, p, 'b-', lw=2)
            ax.set_title(f'd = {d}, σ = {sigma}', fontsize=11)
            ax.set_xlabel('x', fontsize=10)
            ax.set_ylabel('p(x)', fontsize=10)
            ax.grid(True, alpha=0.3)

        except Exception as e:
            ax.set_title(f'd = {d}, σ = {sigma} | ERROR', fontsize=11)

plt.suptitle('Triple-Well Exact PDF | Varying asymmetry (d) and noise (σ)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('asym_step1_d_sigma.png', dpi=100, bbox_inches='tight')
plt.show()

## 3.3.3 Transfer Learning — Asymmetric Configurations

Direct training baseline across 8 seeds and transfer learning from symmetric ($d=0$) and asymmetric ($d=0.3$) source models. Reproduces Figures 3.11–3.15.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.training import train_model
from src.utils import acc_L2

x_eval = np.linspace(-4.0, 4.0, 801)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x_t    = torch.tensor(x_eval, dtype=torch.float32, device=device).view(-1, 1)

CONFIGS = [
    {'d': 0.3, 'sigma': 2.0, 'label': 'Easy\n(d=0.3, σ=2.0)'},
    {'d': 0.3, 'sigma': 1.5, 'label': 'Medium\n(d=0.3, σ=1.5)'},
    {'d': 0.3, 'sigma': 1.0, 'label': 'Hard\n(d=0.3, σ=1.0)'},
]
SEEDS = [0, 1, 7, 13, 42, 99, 123, 256]

# ─── Step 0: Exact solution plot ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, cfg in zip(axes, CONFIGS):
    prob = TripleWellProblem(a=0.2, b=1.5, c=3.0,
                             d=cfg['d'], sigma=cfg['sigma'],
                             domain=(-4.0, 4.0))
    p = prob.exact_solution(x_eval)

    n = len(x_eval); t = n // 3
    h_left   = p[:t].max()
    h_center = p[t:2*t].max()
    h_right  = p[2*t:].max()

    ax.plot(x_eval, p, 'b-', lw=2.5)
    ax.set_title(
        f'{cfg["label"]}\n'
        f'peaks: L={h_left:.3f}, C={h_center:.3f}, R={h_right:.3f}',
        fontsize=11,
    )
    ax.set_xlabel('x'); ax.set_ylabel('p(x)')
    ax.grid(True, alpha=0.3)

plt.suptitle(
    'Asymmetric Triple-Well: Exact stationary PDF\n'
    'd=0.3, varying σ — from easy to hard',
    fontsize=14, y=1.02
)
plt.tight_layout()
plt.savefig('asym_exact_solutions.png', dpi=100, bbox_inches='tight')
plt.show()

# ─── Step 1: Baseline (no TL), 8 seeds × 3 configs ──────────────────────────

baseline_results = {}

for cfg in CONFIGS:
    key = (cfg['d'], cfg['sigma'])
    prob    = TripleWellProblem(a=0.2, b=1.5, c=3.0,
                                d=cfg['d'], sigma=cfg['sigma'],
                                domain=(-4.0, 4.0))
    p_exact = prob.exact_solution(x_eval)
    baseline_results[key] = {}

    print(f"\n{'='*55}")
    print(f"BASELINE | d={cfg['d']}, σ={cfg['sigma']}")
    print(f"{'='*55}")

    for seed in SEEDS:
        result = train_model(
            prob,
            model_config={'output_transform': 'softplus'},
            optimizer_config={'type': 'adam', 'epochs': 30000, 'lr': 1e-3},
            dx=0.01, seed=seed, device=device,
            penalty_factors={'a1': 1.0, 'a2': 1.0, 'a3': 1.0},
            print_every=99999,
        )
        result['model'].eval()
        with torch.no_grad():
            p_pred = result['model'](x_t).cpu().numpy().flatten()

        acc = acc_L2(p_pred, p_exact)
        n = len(x_eval); t = n // 3
        h_left   = p_pred[:t].max()
        h_center = p_pred[t:2*t].max()
        h_right  = p_pred[2*t:].max()
        h_max    = max(h_left, h_center, h_right)

        peaks = sum([
            h_left   > 0.2 * h_max and h_left   > 0.01,
            h_center > 0.2 * h_max and h_center > 0.01,
            h_right  > 0.2 * h_max and h_right  > 0.01,
        ])
        baseline_results[key][seed] = {
            'p': p_pred, 'acc': acc, 'peaks': peaks,
        }
        print(f"  seed={seed}: acc={acc:.4f} | peaks={peaks}/3 | "
              f"L={h_left:.3f} C={h_center:.3f} R={h_right:.3f}")

# Summary baseline
print(f"\n{'='*55}\nBASELINE SUMMARY\n{'='*55}")
for cfg in CONFIGS:
    key = (cfg['d'], cfg['sigma'])
    n_success = sum(1 for s in SEEDS
                    if baseline_results[key][s]['peaks'] == 3
                    and baseline_results[key][s]['acc'] > 0.01)
    mean_acc = np.mean([baseline_results[key][s]['acc'] for s in SEEDS])
    print(f"d={cfg['d']}, σ={cfg['sigma']}: "
          f"{n_success}/{len(SEEDS)} success | mean acc={mean_acc:.4f}")

# ─── Step 2: TL from symmetric source (d=0, σ=2.0) ──────────────────────────

print(f"\n{'='*55}")
print("Training symmetric source: d=0, σ=2.0")
print(f"{'='*55}")

prob_sym = TripleWellProblem(a=0.2, b=1.5, c=3.0,
                             d=0.0, sigma=2.0, domain=(-4.0, 4.0))
result_sym = train_model(
    prob_sym,
    model_config={'output_transform': 'softplus'},
    optimizer_config={'type': 'adam', 'epochs': 30000, 'lr': 1e-3},
    dx=0.01, seed=256, device=device, print_every=99999,
    use_normalization=True,
    penalty_factors={'a1': 1.0, 'a2': 1.0, 'a3': 1.0},
)
source_sym = result_sym['model']
source_sym.eval()
with torch.no_grad():
    p_src = source_sym(x_t).cpu().numpy().flatten()
acc_src = acc_L2(p_src, prob_sym.exact_solution(x_eval))
print(f"Symmetric source acc={acc_src:.4f}")

tl_sym_results = {}
for cfg in CONFIGS:
    key = (cfg['d'], cfg['sigma'])
    prob    = TripleWellProblem(a=0.2, b=1.5, c=3.0,
                                d=cfg['d'], sigma=cfg['sigma'],
                                domain=(-4.0, 4.0))
    p_exact = prob.exact_solution(x_eval)

    print(f"\n[TL sym] d=0,σ=2.0 → d={cfg['d']},σ={cfg['sigma']}")
    result = train_model(
        prob,
        model_config={'output_transform': 'softplus'},
        optimizer_config={'type': 'adam', 'epochs': 30000, 'lr': 1e-3},
        dx=0.01, seed=0, device=device,
        use_normalization=True,
        penalty_factors={'a1': 1.0, 'a2': 1.0, 'a3': 1.0},
        init_model=source_sym, print_every=99999,
    )
    result['model'].eval()
    with torch.no_grad():
        p_pred = result['model'](x_t).cpu().numpy().flatten()

    acc = acc_L2(p_pred, p_exact)
    tl_sym_results[key] = {
        'p': p_pred, 'acc': acc, 'history': result['history'],
    }
    print(f"  -> acc={acc:.4f}")

# ─── Step 3: TL from asymmetric source (d=0.3, σ=2.0) ───────────────────────

print(f"\n{'='*55}")
print("Training asymmetric source: d=0.3, σ=2.0")
print(f"{'='*55}")

prob_asym_src = TripleWellProblem(a=0.2, b=1.5, c=3.0,
                                  d=0.3, sigma=2.0, domain=(-4.0, 4.0))

key_easy = (0.3, 2.0)
best_seed_easy = max(SEEDS, key=lambda s: baseline_results[key_easy][s]['acc'])
print(f"Best baseline seed for d=0.3,σ=2.0: "
      f"seed={best_seed_easy}, "
      f"acc={baseline_results[key_easy][best_seed_easy]['acc']:.4f}")

result_asym_src = train_model(
    prob_asym_src,
    model_config={'output_transform': 'softplus'},
    optimizer_config={'type': 'adam', 'epochs': 30000, 'lr': 1e-3},
    dx=0.01, seed=best_seed_easy, device=device,
    use_normalization=True,
    penalty_factors={'a1': 1.0, 'a2': 1.0, 'a3': 1.0},
    print_every=99999,
)
source_asym = result_asym_src['model']
source_asym.eval()
with torch.no_grad():
    p_src_a = source_asym(x_t).cpu().numpy().flatten()
acc_src_a = acc_L2(p_src_a, prob_asym_src.exact_solution(x_eval))
print(f"Asymmetric source acc={acc_src_a:.4f}")

tl_asym_results = {}
    key = (cfg['d'], cfg['sigma'])
    prob    = TripleWellProblem(a=0.2, b=1.5, c=3.0,
                                d=cfg['d'], sigma=cfg['sigma'],
                                domain=(-4.0, 4.0))
    p_exact = prob.exact_solution(x_eval)

    print(f"\n[TL asym] d=0.3,σ=2.0 → d={cfg['d']},σ={cfg['sigma']}")
    result = train_model(
        prob,
        model_config={'output_transform': 'softplus'},
        optimizer_config={'type': 'adam', 'epochs': 30000, 'lr': 1e-3},
        dx=0.01, seed=0, device=device,
        use_normalization=True,
        penalty_factors={'a1': 1.0, 'a2': 1.0, 'a3': 1.0},
        init_model=source_asym, print_every=99999,
    )
    result['model'].eval()
    with torch.no_grad():
        p_pred = result['model'](x_t).cpu().numpy().flatten()

    acc = acc_L2(p_pred, p_exact)
    tl_asym_results[key] = {
        'p': p_pred, 'acc': acc, 'history': result['history'],
    }
    print(f"  -> acc={acc:.4f}")

# ─── Step 4: Final comparison plots ──────────────────────────────────────────

for cfg in CONFIGS:
    key = (cfg['d'], cfg['sigma'])
    prob    = TripleWellProblem(a=0.2, b=1.5, c=3.0,
                                d=cfg['d'], sigma=cfg['sigma'],
                                domain=(-4.0, 4.0))
    p_exact = prob.exact_solution(x_eval)

    # Worst baseline seed
    worst_seed = min(
        [s for s in SEEDS if baseline_results[key][s]['acc'] > 0.001],
        key=lambda s: baseline_results[key][s]['acc'],
        default=SEEDS[0],
    )
    bl = baseline_results[key][worst_seed]

    n_cols = 3 if key not in tl_asym_results else 4
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))

    # Panel 1: Without TL
    ax = axes[0]
    ax.plot(x_eval, p_exact, 'k-', lw=2, label='Exact solution')
    ax.plot(x_eval, bl['p'], color='#e74c3c', ls='--', lw=2,
            label=f"No TL (seed={worst_seed})\nacc={bl['acc']:.4f}")
    ax.set_title(f'Without TL\nd={cfg["d"]}, σ={cfg["sigma"]}', fontsize=12)
    ax.set_xlabel('x'); ax.set_ylabel('p(x)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    # Panel 2: TL from symmetric source
    ax = axes[1]
    tl_s = tl_sym_results[key]
    ax.plot(x_eval, p_exact, 'k-', lw=2, label='Exact solution')
    ax.plot(x_eval, tl_s['p'], color='#27ae60', ls='--', lw=2,
            label=f"TL from d=0,σ=2.0\nacc={tl_s['acc']:.4f}")
    ax.set_title('TL from symmetric source\n(d=0, σ=2.0)', fontsize=12)
    ax.set_xlabel('x'); ax.set_ylabel('p(x)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    # Panel 3: TL from asymmetric source (if available)
    if key in tl_asym_results:
        ax = axes[2]
        tl_a = tl_asym_results[key]
        ax.plot(x_eval, p_exact, 'k-', lw=2, label='Exact solution')
        ax.plot(x_eval, tl_a['p'], color='#2980b9', ls='--', lw=2,
                label=f"TL from d=0.3,σ=2.0\nacc={tl_a['acc']:.4f}")
        ax.set_title('TL from asymmetric source\n(d=0.3, σ=2.0)', fontsize=12)
        ax.set_xlabel('x'); ax.set_ylabel('p(x)')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
        ax_loss = axes[3]
    else:
        ax_loss = axes[2]

    # Loss convergence (symmetric TL)
    h  = tl_sym_results[key]['history']
    ep = range(len(h['total']))
    ax_loss.semilogy(ep, h['total'], 'k-',  lw=1.5, label='Total')
    ax_loss.semilogy(ep, h['pde'],   'b--', lw=1.5, label='PDE')
    ax_loss.semilogy(ep, h['norm'],  'r--', lw=1.5, label='Norm')
    ax_loss.semilogy(ep, h['bound'], 'g--', lw=1.5, label='Boundary')
    ax_loss.set_title(
        f'Loss convergence (TL sym source)\n'
        f'Final PDE loss = {h["pde"][-1]:.2e}',
        fontsize=12,
    )
    ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Log(Loss)')
    ax_loss.legend(fontsize=9); ax_loss.grid(True, alpha=0.3)

    plt.suptitle(
        f'Experiment 8: Asymmetric Triple-Well | d={cfg["d"]}, σ={cfg["sigma"]}',
        fontsize=14, y=1.02
    )
    plt.tight_layout()
    fname = (f'exp8_asym_d{str(cfg["d"]).replace(".", "")}'
             f'_s{str(cfg["sigma"]).replace(".", "")}.png')
    plt.savefig(fname, dpi=100, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

# ─── Final summary ────────────────────────────────────────────────────────────

print(f"\n{'='*65}")
print("FINAL SUMMARY: Asymmetric Triple-Well")
print(f"{'='*65}")
print(f"{'Config':>20} | {'No TL (best)':>13} | {'TL sym':>8} | {'TL asym':>9}")
print("-" * 65)
for cfg in CONFIGS:
    key = (cfg['d'], cfg['sigma'])
    best_bl = max(baseline_results[key][s]['acc'] for s in SEEDS)
    tl_s_acc = tl_sym_results[key]['acc']
    tl_a_acc = tl_asym_results[key]['acc'] if key in tl_asym_results else '-'
    tl_a_str = f"{tl_a_acc:.4f}" if isinstance(tl_a_acc, float) else tl_a_acc
    print(f"d={cfg['d']}, σ={cfg['sigma']:>5} | {best_bl:>13.4f} | "
          f"{tl_s_acc:>8.4f} | {tl_a_str:>9}")

### Direct Training Results: All Seeds

Reproduces Figure 3.12.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, cfg in zip(axes, CONFIGS):
    key = (cfg['d'], cfg['sigma'])
    prob    = TripleWellProblem(a=0.2, b=1.5, c=3.0,
                                d=cfg['d'], sigma=cfg['sigma'],
                                domain=(-4.0, 4.0))
    p_exact = prob.exact_solution(x_eval)

    ax.plot(x_eval, p_exact, 'k-', lw=2.5, label='Exact solution', zorder=5)

    colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(SEEDS)))
    for seed, color in zip(SEEDS, colors):
        r = baseline_results[key][seed]
        if r['acc'] > 0.01:  # skip trivial p≡0 solutions
            ax.plot(x_eval, r['p'], '-', color=color,
                    lw=1.0, alpha=0.7,
                    label=f"seed={seed} acc={r['acc']:.3f}")

    n_success = sum(1 for s in SEEDS
                    if baseline_results[key][s]['peaks'] == 3
                    and baseline_results[key][s]['acc'] > 0.01)
    ax.set_title(
        f'{cfg["label"]}\nSuccess: {n_success}/{len(SEEDS)}',
        fontsize=11,
    )
    ax.set_xlabel('x'); ax.set_ylabel('p(x)')
    ax.legend(fontsize=6, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle(
    'Experiment 8: Baseline DL-FP | Asymmetric Triple-Well | 8 seeds\n'
    'd=0.3, varying σ — Adam 30000 epochs, no Transfer Learning',
    fontsize=12,
)
plt.tight_layout()
plt.savefig('exp8_baseline_all_seeds.png', dpi=100, bbox_inches='tight')
plt.show()